# KPI de Participación (punto 6.2)

Calcula los KPI de participación a partir de `data/integrado/`. Las funciones son puras (reciben y devuelven DataFrames); la lectura y escritura de archivos se hace solo en las celdas de carga y exportación.

KPI de este notebook:
- Promedio de sesiones por estudiante
- Tiempo promedio en la plataforma
- Porcentaje de cobertura de participación
- Estudiantes activos vs. inactivos

## 1. Carga de datos

In [65]:
from pathlib import Path
import pandas as pd

DATA = Path("../../data")
df_trazabilidad = pd.read_parquet(DATA / "integrado" / "trazabilidad_integrado.parquet")
df_participacion = pd.read_parquet(DATA / "integrado" / "participacion_integrado.parquet")
df_cruce = pd.read_parquet(DATA / "integrado" / "cruce_matriculados_integrado.parquet")

## 2. Promedio de sesiones por estudiante

El log de trazabilidad no trae un identificador de sesión, por lo que se reconstruye: una sesión es una secuencia de eventos de un usuario en una asignatura, y una pausa mayor a `UMBRAL_SESION` abre una sesión nueva. Se cuentan también los matriculados sin actividad (0 sesiones).

In [66]:
# Una sesión termina cuando el estudiante pasa más de este tiempo sin generar eventos.
UMBRAL_SESION = pd.Timedelta(minutes=30)

In [67]:
def sesiones_por_estudiante(
    trazabilidad: pd.DataFrame,
    cruce: pd.DataFrame,
    umbral: pd.Timedelta = UMBRAL_SESION,
) -> pd.DataFrame:
    """Cantidad de sesiones de cada estudiante matriculado, por asignatura.

    Una sesión es una secuencia de eventos de un mismo usuario en una misma
    asignatura donde entre eventos consecutivos pasan como máximo `umbral`.
    El log de trazabilidad no trae un identificador de sesión, por eso se
    reconstruye con este criterio.

    Incluye a los matriculados sin actividad con 0 sesiones, para que no
    inflen el promedio quienes nunca ingresaron.

    Devuelve: materia, usuario, sesiones.
    """
    eventos = trazabilidad.sort_values(["materia", "usuario", "fecha"])
    brecha = eventos.groupby(["materia", "usuario"])["fecha"].diff()
    # Un evento abre sesión nueva si es el primero del usuario (NaT) o si superó el umbral.
    eventos = eventos.assign(nueva_sesion=brecha.isna() | (brecha > umbral))
    sesiones = (
        eventos.groupby(["materia", "usuario"])["nueva_sesion"].sum().astype(int).rename("sesiones")
    )

    matriculados = cruce.loc[cruce["matriculado"], ["materia", "usuario"]]
    return (
        matriculados.merge(sesiones.reset_index(), on=["materia", "usuario"], how="left")
        .fillna({"sesiones": 0})
        .astype({"sesiones": int})
    )

In [68]:
def sesiones_promedio_por_estudiante(
    trazabilidad: pd.DataFrame,
    cruce: pd.DataFrame,
    umbral: pd.Timedelta = UMBRAL_SESION,
) -> pd.DataFrame:
    """KPI: promedio de sesiones por estudiante, por asignatura.

    Justificación: mide la constancia con que los estudiantes vuelven a la
    plataforma. A diferencia del total de interacciones (que premia a quien
    hace muchos clicks en una sola visita), distingue entre ingresos
    esporádicos y un hábito sostenido de estudio.

    Devuelve una fila por asignatura:
      - promedio_matriculados: sobre todos los matriculados (incluye inactivos).
      - promedio_activos: solo sobre quienes tuvieron al menos una sesión.
      - mediana_matriculados: robusta a estudiantes con uso extremo.
    """
    por_estudiante = sesiones_por_estudiante(trazabilidad, cruce, umbral)
    activos = por_estudiante[por_estudiante["sesiones"] > 0]
    return pd.DataFrame(
        {
            "matriculados": por_estudiante.groupby("materia")["usuario"].nunique(),
            "activos": activos.groupby("materia")["usuario"].nunique(),
            "promedio_matriculados": por_estudiante.groupby("materia")["sesiones"].mean(),
            "promedio_activos": activos.groupby("materia")["sesiones"].mean(),
            "mediana_matriculados": por_estudiante.groupby("materia")["sesiones"].median(),
        }
    ).round(2).reset_index()

In [69]:
kpi_sesiones = sesiones_promedio_por_estudiante(df_trazabilidad, df_cruce)
kpi_sesiones

,materia,matriculados,activos,promedio_matriculados,promedio_activos,mediana_matriculados
0,algebra,165,163,31.34,31.72,29.0
1,algoritmos,167,165,27.47,27.80,23.0
2,probabilidad,167,164,39.78,40.51,36.0


**Sensibilidad al umbral:** el resultado depende del criterio de sesión.

In [70]:
pd.DataFrame({
    f"{m} min": sesiones_promedio_por_estudiante(df_trazabilidad, df_cruce, pd.Timedelta(minutes=m)).set_index("materia")["promedio_matriculados"]
    for m in (15, 30, 60)
})

,15 min,30 min,60 min
materia,,,
algebra,34.61,31.34,28.40
algoritmos,30.87,27.47,24.84
probabilidad,45.29,39.78,34.96


## 3. Tiempo promedio en la plataforma

El reporte de participación tiene una fila por estudiante y asignatura con el tiempo acumulado (`tiempo_estimado_de_uso_min`), por lo que se promedia directamente.

In [71]:
def tiempo_por_estudiante(participacion: pd.DataFrame, cruce: pd.DataFrame) -> pd.DataFrame:
    """Minutos de uso estimado de cada estudiante matriculado, por asignatura.

    El reporte de participación tiene una fila por estudiante y asignatura,
    con el tiempo ya acumulado (`tiempo_estimado_de_uso_min`). Los matriculados
    que no figuran en el reporte quedan con 0 minutos.

    Devuelve: materia, usuario, tiempo_min.
    """
    matriculados = cruce.loc[cruce["matriculado"], ["materia", "usuario"]]
    tiempo = participacion[["materia", "usuario", "tiempo_estimado_de_uso_min"]].rename(
        columns={"tiempo_estimado_de_uso_min": "tiempo_min"}
    )
    return matriculados.merge(tiempo, on=["materia", "usuario"], how="left").fillna({"tiempo_min": 0})

In [72]:
def tiempo_promedio_en_plataforma(participacion: pd.DataFrame, cruce: pd.DataFrame) -> pd.DataFrame:
    """KPI: tiempo promedio en la plataforma por estudiante, por asignatura.

    Justificación: complementa a las sesiones. Dos estudiantes con las mismas
    sesiones pueden dedicar tiempos muy distintos; el tiempo acerca la medida
    a la dedicación real al estudio.

    Se promedia el tiempo acumulado de cada estudiante (una fila por alumno).
    Devuelve una fila por asignatura con promedio (matriculados y activos) y
    mediana, en minutos. La distribución es asimétrica (pocos estudiantes con
    uso muy alto), por eso se incluye la mediana.
    """
    por_estudiante = tiempo_por_estudiante(participacion, cruce)
    activos = por_estudiante[por_estudiante["tiempo_min"] > 0]
    return pd.DataFrame(
        {
            "matriculados": por_estudiante.groupby("materia")["usuario"].nunique(),
            "activos": activos.groupby("materia")["usuario"].nunique(),
            "promedio_min_matriculados": por_estudiante.groupby("materia")["tiempo_min"].mean(),
            "promedio_min_activos": activos.groupby("materia")["tiempo_min"].mean(),
            "mediana_min_matriculados": por_estudiante.groupby("materia")["tiempo_min"].median(),
        }
    ).round(2).reset_index()

In [73]:
kpi_tiempo = tiempo_promedio_en_plataforma(df_participacion, df_cruce)
kpi_tiempo

,materia,matriculados,activos,promedio_min_matriculados,promedio_min_activos,mediana_min_matriculados
0,algebra,165,157,229.01,240.68,217.0
1,algoritmos,167,162,206.22,212.58,171.0
2,probabilidad,167,163,406.32,416.29,370.0


## 4. Cobertura de participación

Porcentaje de alumnos cuya participación supera un umbral, sobre el total de alumnos de la asignatura. El total es la unión de los **matriculados** y los **no matriculados con al menos una interacción** (el reporte de participación lista a todo el padrón en cada aula, por lo que quien no está matriculado y nunca participó no se cuenta). El umbral es variable: con 0 mide quién participó al menos una vez.

In [74]:
def cobertura_participacion(
    participacion: pd.DataFrame,
    cruce: pd.DataFrame,
    umbral: float = 0,
    columna: str = "interacciones",
) -> pd.DataFrame:
    """KPI: porcentaje de cobertura de participación, por asignatura.

    Cobertura = alumnos con `columna` > `umbral` / total de alumnos * 100.

    El total de alumnos de una asignatura es la unión de:
      - los matriculados en la asignatura (aunque no hayan participado), y
      - los no matriculados que tuvieron al menos una interacción en el aula.
    Así no se cuenta a quienes figuran en el reporte con 0 interacciones y sin
    matrícula (el reporte lista a todo el padrón de la plataforma en cada aula).

    El total no depende del umbral: solo el numerador cambia. Con umbral=0 mide
    quién participó al menos una vez; subiéndolo mide quién participó con cierta
    intensidad (p. ej. más de 100 interacciones).

    Parámetros:
      - umbral: valor que la participación debe superar (estricto).
      - columna: métrica de participación a evaluar ("interacciones",
        "tiempo_estimado_de_uso_min", "avance", ...).

    Devuelve una fila por asignatura: total_alumnos, matriculados,
    no_matriculados_con_actividad, con_participacion, cobertura_pct.
    """
    claves = ["materia", "usuario"]
    matriculados = cruce.loc[cruce["matriculado"], claves]
    con_actividad = participacion.loc[participacion["interacciones"] > 0, claves]
    universo = pd.concat([matriculados, con_actividad]).drop_duplicates()

    universo = universo.merge(
        participacion[claves + [columna]], on=claves, how="left"
    ).fillna({columna: 0})
    universo["es_matriculado"] = universo.set_index(claves).index.isin(matriculados.set_index(claves).index)
    universo["participa"] = universo[columna] > umbral

    resumen = universo.groupby("materia").agg(
        total_alumnos=("usuario", "nunique"),
        matriculados=("es_matriculado", "sum"),
        con_participacion=("participa", "sum"),
    )
    resumen["no_matriculados_con_actividad"] = resumen["total_alumnos"] - resumen["matriculados"]
    resumen["cobertura_pct"] = (100 * resumen["con_participacion"] / resumen["total_alumnos"]).round(2)
    return resumen[
        ["total_alumnos", "matriculados", "no_matriculados_con_actividad", "con_participacion", "cobertura_pct"]
    ].reset_index()

In [75]:
UMBRAL_PARTICIPACION = 0  # interacciones que hay que superar para contar como participante
kpi_cobertura = cobertura_participacion(df_participacion, df_cruce, UMBRAL_PARTICIPACION)
kpi_cobertura

,materia,total_alumnos,matriculados,no_matriculados_con_actividad,con_participacion,cobertura_pct
0,algebra,270,165,105,268,99.26
1,algoritmos,278,167,111,276,99.28
2,probabilidad,276,167,109,273,98.91


**Sensibilidad al umbral:** cobertura (%) según las interacciones mínimas exigidas.

In [76]:
pd.concat(
    {u: cobertura_participacion(df_participacion, df_cruce, u).set_index("materia")["cobertura_pct"] for u in (0, 50, 100, 250, 500)},
    axis=1,
).rename_axis(columns="umbral (interacciones)")

umbral (interacciones),0,50,100,250,500
materia,,,,,
algebra,99.26,75.19,68.15,36.67,4.81
algoritmos,99.28,78.42,65.47,34.17,3.60
probabilidad,98.91,78.99,68.48,48.55,18.12


## 5. Estudiantes activos vs. inactivos

Un estudiante está **activo** si su última interacción fue hace `N` días o menos respecto de la fecha de corte de los datos, e **inactivo** si pasaron más de `N` días o si nunca interactuó. Universo: matriculados y no matriculados con actividad (igual que la cobertura). Se parte de N = 15.

In [77]:
def estudiantes_activos_vs_inactivos(
    participacion: pd.DataFrame,
    cruce: pd.DataFrame,
    dias: int = 15,
    fecha_corte: pd.Timestamp | None = None,
) -> pd.DataFrame:
    """KPI: estudiantes activos vs. inactivos, por asignatura.

    Un estudiante es activo si su última interacción ocurrió dentro de los
    últimos `dias` días respecto de la fecha de corte; inactivo si pasaron más
    de `dias` días o si nunca interactuó (sin última interacción).

    Justificación: a diferencia de la cobertura (si participó alguna vez),
    mide el estado reciente y permite detectar a tiempo a quienes se alejaron
    de la cursada.

    Universo: igual que en la cobertura, los matriculados más los no
    matriculados con al menos una interacción.

    Parámetros:
      - dias: ventana de actividad N. Inactivo si pasaron más de N días.
      - fecha_corte: fecha de referencia. Por defecto, la última interacción
        registrada en todo el reporte (la fecha del corte de los datos), para
        que el resultado no dependa de cuándo se ejecute el notebook.

    Devuelve una fila por asignatura: total_alumnos, activos, inactivos,
    pct_activos, pct_inactivos, fecha_corte.
    """
    claves = ["materia", "usuario"]
    if fecha_corte is None:
        fecha_corte = participacion["ultima_interaccion"].max()

    matriculados = cruce.loc[cruce["matriculado"], claves]
    con_actividad = participacion.loc[participacion["interacciones"] > 0, claves]
    universo = pd.concat([matriculados, con_actividad]).drop_duplicates()
    universo = universo.merge(participacion[claves + ["ultima_interaccion"]], on=claves, how="left")

    dias_sin_interaccion = (fecha_corte - universo["ultima_interaccion"]).dt.total_seconds() / 86400
    # NaN (nunca interactuó) compara como False -> inactivo
    universo["activo"] = dias_sin_interaccion <= dias

    resumen = universo.groupby("materia")["activo"].agg(total_alumnos="size", activos="sum")
    resumen["inactivos"] = resumen["total_alumnos"] - resumen["activos"]
    resumen["pct_activos"] = (100 * resumen["activos"] / resumen["total_alumnos"]).round(2)
    resumen["pct_inactivos"] = (100 - resumen["pct_activos"]).round(2)
    resumen["fecha_corte"] = fecha_corte.normalize()
    return resumen.reset_index()

In [78]:
DIAS_ACTIVIDAD = 15  # N: días máximos sin interacción para seguir considerándose activo
kpi_activos = estudiantes_activos_vs_inactivos(df_participacion, df_cruce, DIAS_ACTIVIDAD)
kpi_activos

,materia,total_alumnos,activos,inactivos,pct_activos,pct_inactivos,fecha_corte
0,algebra,270,176,94,65.19,34.81,2025-10-24
1,algoritmos,278,151,127,54.32,45.68,2025-10-24
2,probabilidad,276,164,112,59.42,40.58,2025-10-24


**Sensibilidad a N:** porcentaje de activos según la ventana elegida.

In [80]:
pd.concat(
    {n: estudiantes_activos_vs_inactivos(df_participacion, df_cruce, n).set_index("materia")["pct_activos"] for n in (5, 7, 15, 30, 60)},
    axis=1,
).rename_axis(columns="N (días)")

N (días),5,7,15,30,60
materia,,,,,
algebra,39.26,42.22,65.19,69.63,80.74
algoritmos,34.89,37.77,54.32,62.59,77.34
probabilidad,42.39,43.84,59.42,73.91,81.16
